In [1]:
# Проверка порта
import socket
sock = socket.create_connection(("data-flow-postgres-airflow-1", 5432), timeout=3)
print("✅ Порт 5432 доступен")
sock.close()

✅ Порт 5432 доступен


In [2]:
from pyspark.sql import SparkSession

# 1. Создаём сессию с JDBC-драйвером
spark = (SparkSession.builder
    .appName("DE_Task_PG_Connection")
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3")
    .config("spark.executor.instances", "2")
    .getOrCreate())

# 2. Параметры подключения
jdbc_url = "jdbc:postgresql://postgres-source:5432/source"

props = {
    "user": "source",
    "password": "source",  # ← замени на реальный из docker-compose
    "driver": "org.postgresql.Driver"
}

# 3. Чтение таблицы
df = spark.read.jdbc(url=jdbc_url, table="d_raw.raw_events", properties=props)

# 4. Вывод таблицы
print(f"✅ Загружено строк: {df.count()}")
df.show(5, truncate=False)

✅ Загружено строк: 1000
+--------+-------+-----------+--------------------------+
|event_id|user_id|event_type |created_at                |
+--------+-------+-----------+--------------------------+
|1       |100001 |view_item  |2026-05-06 07:54:23.546406|
|2       |100002 |add_to_cart|2026-05-06 07:55:23.546406|
|3       |100003 |checkout   |2026-05-06 07:56:23.546406|
|4       |100004 |logout     |2026-05-06 07:57:23.546406|
|5       |100005 |login      |2026-05-06 07:58:23.546406|
+--------+-------+-----------+--------------------------+
only showing top 5 rows



In [3]:
df.createOrReplaceTempView("raw_events")

df1 = spark.sql("""
SELECT *
FROM raw_events
""")

df1.show()

+--------+-------+-----------+--------------------+
|event_id|user_id| event_type|          created_at|
+--------+-------+-----------+--------------------+
|       1| 100001|  view_item|2026-05-06 07:54:...|
|       2| 100002|add_to_cart|2026-05-06 07:55:...|
|       3| 100003|   checkout|2026-05-06 07:56:...|
|       4| 100004|     logout|2026-05-06 07:57:...|
|       5| 100005|      login|2026-05-06 07:58:...|
|       6| 100006|  view_item|2026-05-06 07:59:...|
|       7| 100007|add_to_cart|2026-05-06 08:00:...|
|       8| 100008|   checkout|2026-05-06 08:01:...|
|       9| 100009|     logout|2026-05-06 08:02:...|
|      10| 100010|      login|2026-05-06 08:03:...|
|      11| 100011|  view_item|2026-05-06 08:04:...|
|      12| 100012|add_to_cart|2026-05-06 08:05:...|
|      13| 100013|   checkout|2026-05-06 08:06:...|
|      14| 100014|     logout|2026-05-06 08:07:...|
|      15| 100015|      login|2026-05-06 08:08:...|
|      16| 100016|  view_item|2026-05-06 08:09:...|
|      17| 1

In [ ]:
df1.show()

In [ ]:
spark.stop()